# Data Availability: Which ISMN Stations Are Covered by BIOMASS L1C?

This notebook takes a GeoJSON file of points (here: ISMN soil moisture stations, `ISMN_data/ismn_locations_overlap.geojson`) and queries the ESA MAAP STAC catalogue to find out, for each point, whether **BIOMASS Level 1C** products (`BiomassLevel1c` collection) are available **at that location and within that station's measurement period** (`start` / `end` attributes).

It then checks the station time series in `ISMN_data/temporal/` (`<network>+<station>.csv`) and attaches to every BIOMASS product the **nearest good-quality near-surface soil moisture** value, so the stations usable for a BIOMASS–soil moisture analysis can be selected.

It is adapted from *Biomass Data Access Example.ipynb*. Only catalogue discovery is used, so **no access token is needed**.

Outputs:
- a per-point summary, where each point gets one of three statuses:
  - `in_period`: BIOMASS L1C products overlap the measurement period
  - `outside_period`: the point is covered, but only outside the measurement period
  - `no_coverage`: no L1C product covers the point at all
- a soil moisture status per point (`sm_status`): `no_sm_file`, `no_good_sm`, `sm_not_at_acquisitions`, `sm_at_acquisitions`
- a per-product table (one row per point × product) with an `in_period` flag and the matched soil moisture (`sm_surface`, `sm_dt_min`)
- an interactive map of the points and the matching footprints
- CSV / GeoJSON exports of the above

### 1. Packages and Libraries

In [ ]:
from pystac_client import Client
from shapely.geometry import shape
from folium import GeoJson
import geopandas as gpd
from tqdm import tqdm
import pandas as pd
import pathlib
import folium

### 2. Parameters

- `GEOJSON_PATH`: your point file. Any CRS is fine; it is reprojected to EPSG:4326. MultiPoints are exploded into single points.
- `ID_FIELDS`: one or more attributes that together identify each point. They are joined with `/`. If `None`, the row order is used.
- `START_FIELD` / `END_FIELD`: the attributes holding each point's period of interest. Timestamps without a timezone are read as UTC. A missing value means the period is open on that side. Set both to `None` to ignore time.
- `TIME_BUFFER_DAYS`: widens every point's period by this many days on both sides.
- `DATETIME`: an optional global window applied to the catalogue search, e.g. `"2026-01-01T00:00:00Z/2026-07-01T23:59:59Z"`. `None` searches the whole archive.
- `OUTPUT_DIR`: folder where outputs are written, e.g. `/home/jovyan/test_data`; results go to its `l1c_availability/` subfolder, which *Biomass L1C Point Extraction.ipynb* reads. It is created if missing.
- `SM_DIR`: folder with the ISMN time series, one CSV per station named `<network>+<station>.csv` (long format, one row per sensor and hour; depth in the sensor name, e.g. `..._soil_moisture_0.050000_0.050000`).
- `SM_MAX_DEPTH_M`: sensors whose lower depth is ≤ this value are averaged into the near-surface soil moisture `sm_surface` [m³/m³].
- `SM_GOOD_FLAGS`: ISMN quality flags accepted (`G` = good). Other flags (e.g. `D02`/`D01` frozen soil, `C02`/`C03` out of range) are discarded.
- `SM_TOLERANCE`: maximum time difference between a BIOMASS acquisition and the matched (hourly) soil moisture value.
- `PRODUCT_TYPES`: `None` returns all L1C types. Otherwise give a list, e.g. `["S1_STA__1S"]`. The L1C product types are listed in the [L1C queryables](https://catalog.maap.eo.esa.int/catalogue/collections/BiomassLevel1c/queryables): `S1_SCS__1S`, `S1_STA__1S`, `S2_STA__1S`, `S3_SCS__1S`, `S3_STA__1S`.

In [ ]:
GEOJSON_PATH = "ISMN_data/ismn_locations_overlap.geojson"
ID_FIELDS = ["network", "station"]
START_FIELD = "start"
END_FIELD = "end"
TIME_BUFFER_DAYS = 0
COLLECTION = "BiomassLevel1c"
DATETIME = None                   # None = all available dates
PRODUCT_TYPES = None              # None = all L1C product types, or e.g. ["S1_STA__1S"]
OUTPUT_DIR = pathlib.Path("/home/jovyan/test_data")   # <- where all outputs are written
AVAILABILITY_DIR = OUTPUT_DIR / "l1c_availability"
SM_DIR = pathlib.Path("ISMN_data/temporal")
SM_MAX_DEPTH_M = 0.10
SM_GOOD_FLAGS = ["G"]
SM_TOLERANCE = pd.Timedelta("1h")

### 3. Load the points

In [ ]:
points = gpd.read_file(GEOJSON_PATH)
points = points.to_crs("EPSG:4326") if points.crs else points.set_crs("EPSG:4326")
points = points.explode(index_parts=False).reset_index(drop=True)

bad = ~points.geom_type.eq("Point")
if bad.any():
    raise ValueError(f"{bad.sum()} non-point geometries found: {points.geom_type[bad].unique()}")

if ID_FIELDS:
    points["point_id"] = points[ID_FIELDS].astype(str).agg("/".join, axis=1)
else:
    points["point_id"] = points.index.astype(str)
if points["point_id"].duplicated().any():
    raise ValueError("point_id is not unique; add more ID_FIELDS.")


def to_utc(col):
    """Parse a timestamp column; naive timestamps are read as UTC."""
    if col is None or col not in points.columns:
        return pd.Series(pd.NaT, index=points.index, dtype="datetime64[ns, UTC]")
    return pd.to_datetime(points[col], utc=True)


buffer = pd.Timedelta(days=TIME_BUFFER_DAYS)
points["period_start"] = to_utc(START_FIELD) - buffer
points["period_end"] = to_utc(END_FIELD) + buffer

print(f"{len(points)} points loaded.")
points.drop(columns="geometry")

### 4. Query the ESA MAAP STAC catalog for each point

For each point, a STAC search is run with a zero-size bounding box `bbox=[x, y, x, y]`. The catalogue matches it against each product's **footprint**, not just the product's bounding box. A local footprint check is applied on top as a safety net.

The search is spatial only (plus the optional global `DATETIME`). Each point's own period is applied afterwards. A product counts as `in_period` when its acquisition `[start_datetime, end_datetime]` overlaps `[period_start, period_end]`. Running the search this way lets the notebook tell points with no coverage at all apart from points covered only outside their measurement period.

Notes on this catalogue (checked October 2026):
- `intersects` is **ignored with GET**: the whole collection comes back. With POST it filters correctly, but **POST paging is broken**: the next-page request fails, so you would only get the first page. The point bbox with GET avoids both problems.
- The CQL2 `IN` operator is not supported, so several product types are combined with `OR`.

In [ ]:
catalog_url = 'https://catalog.maap.eo.esa.int/catalogue/'
catalog = Client.open(catalog_url)

cql_filter = None
if PRODUCT_TYPES:
    cql_filter = " OR ".join(f"product:type='{t}'" for t in PRODUCT_TYPES)


def search_point(pt):
    """Return all items of COLLECTION whose footprint contains the point."""
    search = catalog.search(
        collections=[COLLECTION],
        bbox=[pt.x, pt.y, pt.x, pt.y],
        datetime=DATETIME,
        filter=cql_filter,
        method="GET",
        limit=100,
    )
    return [it for it in search.items() if it.geometry and shape(it.geometry).intersects(pt)]


rows = []
footprints = {}
for pid, pt in tqdm(zip(points["point_id"], points.geometry), total=len(points)):
    for it in search_point(pt):
        props = it.properties or {}
        ql = it.assets.get("quicklook_jpg") or it.assets.get("thumbnail")
        rows.append({
            "point_id": pid,
            "product_id": it.id,
            "product_type": props.get("product:type"),
            "start_datetime": props.get("start_datetime"),
            "end_datetime": props.get("end_datetime"),
            "orbit_state": props.get("sat:orbit_state"),
            "grid_code": props.get("grid:code"),
            "stack_id": props.get("eofeos:stack_id"),
            "mission_phase": props.get("eofeos:mission_phase"),
            "coreg_primary": props.get("eofeos:is_coregistration_primary"),
            "quicklook": ql.href if ql else None,
        })
        footprints[it.id] = it.geometry

products = pd.DataFrame(rows, columns=[
    "point_id", "product_id", "product_type", "start_datetime", "end_datetime",
    "orbit_state", "grid_code", "stack_id", "mission_phase", "coreg_primary", "quicklook",
])
products["start_datetime"] = pd.to_datetime(products["start_datetime"], utc=True)
products["end_datetime"] = pd.to_datetime(products["end_datetime"], utc=True)

# Temporal criterion: product acquisition overlaps the point's period (open ends allowed)
products = products.merge(points[["point_id", "period_start", "period_end"]], on="point_id")
products["in_period"] = (
    (products["period_end"].isna() | (products["start_datetime"] <= products["period_end"]))
    & (products["period_start"].isna() | (products["end_datetime"] >= products["period_start"]))
)
products = products.sort_values(["point_id", "start_datetime"], ignore_index=True)

print(f"{len(products)} point/product matches, {products['product_id'].nunique()} unique products, "
      f"{products['in_period'].sum()} matches within the point periods.")

### 5. Soil moisture at the BIOMASS acquisitions

For each station with a time series file, good-quality (`SM_GOOD_FLAGS`) soil moisture from sensors not deeper than `SM_MAX_DEPTH_M` is averaged per hour into `sm_surface`. Every BIOMASS product at that station is matched to the nearest hourly value within `SM_TOLERANCE` (`sm_dt_min` = time difference in minutes). ISMN timestamps are UTC.

In [ ]:
def load_sm(network, station):
    """Soil moisture records of one station (all depths and flags), or None if there is no file."""
    f = SM_DIR / f"{network}+{station}.csv"
    if not f.exists():
        return None
    d = pd.read_csv(f, usecols=["sensor", "date_time", "soil_moisture", "soil_moisture_flag"],
                    dtype={"soil_moisture_flag": str})
    d = d[d["sensor"].str.contains("_soil_moisture_") & d["soil_moisture"].notna()].copy()
    depth = d["sensor"].str.rsplit("_", n=2, expand=True)
    d["depth_from"], d["depth_to"] = depth[1].astype(float), depth[2].astype(float)
    d["date_time"] = pd.to_datetime(d["date_time"], utc=True)
    d["good"] = d["soil_moisture_flag"].isin(SM_GOOD_FLAGS)
    return d


sm_rows, matched = [], []
for _, p in tqdm(points.iterrows(), total=len(points)):
    d = load_sm(p["network"], p["station"])
    row = {"point_id": p["point_id"], "sm_file": d is not None}
    prods = products[products["point_id"] == p["point_id"]]
    if d is not None:
        surf = d[d["good"] & (d["depth_to"] <= SM_MAX_DEPTH_M)]
        series = surf.groupby("date_time")["soil_moisture"].mean().rename("sm_surface").reset_index()
        row.update({
            "sm_depths_m": ", ".join(f"{x:g}" for x in sorted(d["depth_to"].unique())),
            "sm_good_fraction": round(d["good"].mean(), 3),
            "sm_first": series["date_time"].min(), "sm_last": series["date_time"].max(),
        })
        if len(prods) and len(series):
            m = pd.merge_asof(prods.sort_values("start_datetime"), series, left_on="start_datetime",
                              right_on="date_time", direction="nearest", tolerance=SM_TOLERANCE)
            m["sm_dt_min"] = (m["date_time"] - m["start_datetime"]).dt.total_seconds() / 60
            matched.append(m[["point_id", "product_id", "sm_surface", "sm_dt_min"]])
    sm_rows.append(row)

sm_summary = pd.DataFrame(sm_rows).reindex(
    columns=["point_id", "sm_file", "sm_depths_m", "sm_good_fraction", "sm_first", "sm_last"])
products = products.drop(columns=["sm_surface", "sm_dt_min"], errors="ignore")
if matched:
    products = products.merge(pd.concat(matched), on=["point_id", "product_id"], how="left")
else:
    products["sm_surface"], products["sm_dt_min"] = float("nan"), float("nan")
products["sm_available"] = products["sm_surface"].notna()

print(f"{sm_summary['sm_file'].sum()} of {len(points)} stations have a soil moisture file; "
      f"{products['sm_available'].sum()} of {len(products)} point/product matches have soil moisture.")

### 6. Per-point availability summary

In [ ]:
def summarise(df):
    return df.groupby("point_id").agg(
        n_products=("product_id", "nunique"),
        n_stacks=("stack_id", "nunique"),
        first_acquisition=("start_datetime", "min"),
        last_acquisition=("start_datetime", "max"),
        product_types=("product_type", lambda s: ", ".join(sorted(s.dropna().unique()))),
        orbit_states=("orbit_state", lambda s: ", ".join(sorted(s.dropna().unique()))),
    )


summary_in = summarise(products[products["in_period"]])
summary_all = summarise(products)[["n_products", "first_acquisition", "last_acquisition"]].add_suffix("_any_time")

n_sm = products[products["sm_available"]].groupby("point_id")["product_id"].nunique().rename("n_products_with_sm")
points_summary = (
    points.merge(summary_in, left_on="point_id", right_index=True, how="left")
          .merge(summary_all, left_on="point_id", right_index=True, how="left")
          .merge(n_sm, left_on="point_id", right_index=True, how="left")
          .merge(sm_summary, on="point_id", how="left")
)
for c in ["n_products", "n_stacks", "n_products_any_time", "n_products_with_sm"]:
    points_summary[c] = points_summary[c].fillna(0).astype(int)

points_summary["status"] = "no_coverage"
points_summary.loc[points_summary["n_products_any_time"] > 0, "status"] = "outside_period"
points_summary.loc[points_summary["n_products"] > 0, "status"] = "in_period"

points_summary["sm_status"] = "no_sm_file"
points_summary.loc[points_summary["sm_file"], "sm_status"] = "no_good_sm"
points_summary.loc[points_summary["sm_first"].notna(), "sm_status"] = "sm_not_at_acquisitions"
points_summary.loc[points_summary["n_products_with_sm"] > 0, "sm_status"] = "sm_at_acquisitions"

print(points_summary["status"].value_counts().to_string(), "\n")
print(pd.crosstab(points_summary["status"], points_summary["sm_status"]).to_string(), "\n")
points_summary[points_summary["sm_file"]].drop(columns="geometry").sort_values(["sm_status", "point_id"])

#### 6.1. Products found within the period of a given point

In [ ]:
in_period_ids = points_summary.loc[points_summary["sm_status"] == "sm_at_acquisitions", "point_id"]
if not len(in_period_ids):
    in_period_ids = points_summary.loc[points_summary["status"] == "in_period", "point_id"]
pid = in_period_ids.iloc[0] if len(in_period_ids) else None
if pid is not None:
    display(products[(products["point_id"] == pid) & products["in_period"]].drop(columns="quicklook"))
else:
    print("No point has L1C data within its period.")

### 7. Visualise points and footprints on a map

Point colours: **green** = L1C within the period, **orange** = covered only outside the period, **red** = no coverage. Stations with soil moisture at BIOMASS acquisitions get a thick **black ring**. Only the footprints of products within the period are drawn. Hover over a footprint to see its product ID.

In [ ]:
STATUS_COLOR = {"in_period": "green", "outside_period": "orange", "no_coverage": "red"}

m = folium.Map(tiles="OpenStreetMap")

fp_ids = products.loc[products["in_period"], "product_id"].unique()
if len(fp_ids):
    fp_gdf = gpd.GeoDataFrame(
        {"product_id": fp_ids},
        geometry=[shape(footprints[i]) for i in fp_ids],
        crs="EPSG:4326",
    ).merge(products.drop_duplicates("product_id")[["product_id", "product_type", "start_datetime"]],
            on="product_id")
    fp_gdf["start_datetime"] = fp_gdf["start_datetime"].astype(str)
    GeoJson(
        fp_gdf.__geo_interface__,
        name=f"L1C footprints in period ({len(fp_gdf)})",
        tooltip=folium.GeoJsonTooltip(fields=["product_id", "product_type", "start_datetime"]),
        style_function=lambda _: {"color": "#ff7f0e", "weight": 1, "fillOpacity": 0.05},
    ).add_to(m)

pts_layer = folium.FeatureGroup(name=f"Points ({len(points_summary)})")
for _, r in points_summary.iterrows():
    folium.CircleMarker(
        location=[r.geometry.y, r.geometry.x],
        radius=6,
        color="black" if r["sm_status"] == "sm_at_acquisitions" else STATUS_COLOR[r["status"]],
        weight=4 if r["sm_status"] == "sm_at_acquisitions" else 1,
        fill=True,
        fill_color=STATUS_COLOR[r["status"]],
        fill_opacity=0.9,
        tooltip=(f"{r['point_id']}: {r['n_products']} L1C in period / "
                 f"{r['n_products_any_time']} any time / {r['n_products_with_sm']} with soil moisture"),
    ).add_to(pts_layer)
pts_layer.add_to(m)

minx, miny, maxx, maxy = points_summary.total_bounds
m.fit_bounds([[miny, minx], [maxy, maxx]])
folium.LayerControl(collapsed=False).add_to(m)
m

### 8. Preview a quicklook

In [ ]:
from IPython.display import Image, display

if pid is not None:
    sel = products[(products["point_id"] == pid) & products["in_period"]].dropna(subset=["quicklook"])
    if len(sel):
        last = sel.iloc[-1]
        print(f"Latest in-period product at point {pid}: {last['product_id']}")
        display(Image(url=last["quicklook"]))

### 9. Save the results

In [ ]:
AVAILABILITY_DIR.mkdir(parents=True, exist_ok=True)

out_points = points_summary.copy()
for c in out_points.columns:
    if pd.api.types.is_datetime64_any_dtype(out_points[c]):
        out_points[c] = out_points[c].astype(str)
out_points.to_file(AVAILABILITY_DIR / "points_l1c_availability.geojson", driver="GeoJSON")
out_points.drop(columns="geometry").to_csv(AVAILABILITY_DIR / "points_l1c_availability.csv", index=False)
products.to_csv(AVAILABILITY_DIR / "points_l1c_products.csv", index=False)

print(f"Saved to {AVAILABILITY_DIR.resolve()}")

To stream or download any of the listed products, take its `product_id` and follow sections 2.6–2.7 of *Biomass Data Access Example.ipynb*. Those steps need a MAAP access token.